# Phase 3.1：计时、Warmup、Mean、P50 和 P95

## 目标

建立一个可复用的 benchmark 计时器，测量真实 `KnowledgeBase.search()`，理解冷启动与稳定态的区别，并输出 mean/P50/P95。

**本课交付：** `data/processed/phase3_timing_baseline.json`。

## Evidence Quest 任务卡：Phase 3.1：检索竞速计时台

**你的身份：** 性能计时工程师  
**案件背景：** 用户说搜索有点慢，但‘感觉快’不是工程证据。你要区分预热、稳定运行、平均延迟和尾延迟。

### 本关专业 Goal

建立可复现的 mean、P50、P95 计时基线。

### 你要交付的作品

**检索速度计时卡**

### 通关判定

- 先运行带逐行中文注释的示范，预测输出，再自己重新敲一遍关键代码。
- 至少改变一个参数或输入，记录它为什么改变了结果。
- 完成末尾的 Boss Challenge，并能解释一个失败样本。
- 把本关产物交给下一关，而不是把代码停留在 Notebook 屏幕上。

**通关奖励：** 解锁徽章：延迟计时员  
**学习节奏：** 看故事 -> 跟敲一小段 -> 观察输出 -> 自己改写 -> 验收作品。

## 1. 性能数字为什么需要实验设计？

一次调用的耗时可能包含首次导入、内存分配、缓存建立等因素。稳定态 benchmark 通常先 warmup，再测多次。平均值描述总体水平，P50 描述典型请求，P95 暴露长尾。

当前语料很小，所以数字不是生产承诺；我们正在验证测量方法是否正确。

In [1]:
# 导入 Path，用它表示跨平台的文件路径。
from pathlib import Path

# 导入 json，用它读取和保存项目的结构化数据。
import json

# 导入 sys，用它把项目根目录加入 Python 的模块搜索路径。
import sys


# 定义一个函数，负责从当前工作目录向上查找项目根目录。
def find_project_root() -> Path:
    # 把当前目录和它的所有父目录放进候选列表。
    candidates = [Path.cwd(), *Path.cwd().parents]

    # 逐个检查候选目录是否包含本项目的两个核心模块目录。
    for candidate in candidates:
        # 找到同时存在的目录时，返回这个候选目录。
        if (candidate / "phase1_doc_parser").is_dir() and (candidate / "phase2_semantic_search").is_dir():
            return candidate

    # 如果所有候选目录都不符合，说明 Jupyter 启动位置不在项目内。
    raise RuntimeError("找不到项目根目录，请从 ai-search-rag-internship 启动 JupyterLab")


# 执行查找函数，得到当前项目根目录。
ROOT = find_project_root()

# 如果项目根目录还不在模块搜索路径中，就把它添加进去。
if str(ROOT) not in sys.path:
    # 把项目根目录插入最前面，确保导入的是当前项目代码。
    sys.path.insert(0, str(ROOT))

# 打印根目录，帮助学习者确认 Notebook 没有在错误目录运行。
print("项目根目录:", ROOT)

项目根目录: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship


In [2]:
# 定义本关任务编号，后面的记录会用它区分不同阶段。
QUEST_STAGE = 'phase3.1'

# 定义学习者可以持续保存的案件档案路径。
QUEST_PROFILE_PATH = ROOT / "data" / "processed" / "evidence_quest_profile.json"

# 如果第一次打开课程还没有档案，就使用一个安全的默认案件。
default_profile = {
    "case_name": "校园知识库失踪案",
    "audience": "需要快速查证资料的同学",
    "must_answer": "证据来自哪里，能否回到原文？",
    "must_refuse": "检索结果没有证据时必须说不知道",
    "xp": 0,
    "badges": [],
}

# 检查任务档案是否已经由 Mission Control 创建。
if QUEST_PROFILE_PATH.is_file():
    # 读取学员自己的案件主题，让所有 Notebook 共享同一个故事。
    quest_profile = json.loads(QUEST_PROFILE_PATH.read_text(encoding="utf-8"))
else:
    # 没有档案时复制默认值，避免直接修改模板字典。
    quest_profile = dict(default_profile)

# 计算当前累计经验值；错误值按 0 处理，避免看板阻塞学习。
quest_xp = int(quest_profile.get("xp", 0))

# 读取已经获得的徽章，并复制成当前 Notebook 的列表。
quest_badges = list(quest_profile.get("badges", []))

# 用可见的文字看板告诉学习者自己正在解决哪个真实问题。
print("Evidence Quest / 当前关卡:", QUEST_STAGE)
print("案件:", quest_profile.get("case_name", default_profile["case_name"]))
print("服务对象:", quest_profile.get("audience", default_profile["audience"]))
print("累计 XP:", quest_xp, "| 徽章:", ", ".join(quest_badges) if quest_badges else "尚未获得")

Evidence Quest / 当前关卡: phase3.1
案件: 校园知识库失踪案
服务对象: 需要复习课程资料的同学
累计 XP: 0 | 徽章: 案件接收员


In [3]:
# 导入 perf_counter，它适合测量短时间间隔。
from time import perf_counter

# 导入统计函数，计算平均值和中位数。
from statistics import mean, median

# 导入 KnowledgeBase，它会复用 Phase 1 和 Phase 2 的生产逻辑。
from phase4_mini_rag_system.knowledge_base import KnowledgeBase

# 创建一个知识库对象。
knowledge_base = KnowledgeBase()

# 使用固定输入和固定分块参数构建索引。
knowledge_base.ingest(ROOT / "phase1_doc_parser" / "examples" / "input", chunk_size=128, overlap=32)

# 输出索引版本，记录 benchmark 的数据条件。
print("index_version:", knowledge_base.index_version)

# 确认索引中存在数据。
assert knowledge_base.chunks

index_version: chunks-4-size-128-overlap-32


In [4]:
# 定义一个函数，计时一次真实搜索并返回毫秒数。
def time_one_search(query, top_k=5):
    # 在计时开始前记录高精度时间点。
    start_time = perf_counter()

    # 执行真正的知识库搜索，不测量空函数。
    knowledge_base.search(query, top_k=top_k)

    # 记录结束时间并转换为毫秒。
    elapsed_ms = (perf_counter() - start_time) * 1000

    # 返回这一次请求的耗时。
    return elapsed_ms

# 测量第一次调用，观察冷启动样本。
first_ms = time_one_search("Chunk overlap")

# 打印第一次调用时间。
print("first call ms:", round(first_ms, 4))

first call ms: 0.1204


In [5]:
# 创建空列表，用于保存 warmup 后的稳定态样本。
stable_samples = []

# 先执行五次不计入结果的 warmup。
for _ in range(5):
    # 让解释器和索引有机会完成首次准备工作。
    time_one_search("Chunk overlap")

# 再执行三十次正式测量。
for _ in range(30):
    # 把每次耗时保存到样本列表。
    stable_samples.append(time_one_search("Chunk overlap"))

# 输出部分样本，确认确实测量了多次。
print("samples:", [round(sample, 4) for sample in stable_samples[:5]], "...")

# 确认样本数量满足预期。
assert len(stable_samples) == 30

samples: [0.0505, 0.0292, 0.03, 0.0264, 0.0416] ...


## 2. 自己实现 percentile

为了理解 P50/P95，不先调用统计库。把样本排序后，用位置取近似分位数。不同工具可能使用不同插值规则，正式报告必须说明采用的定义。

In [6]:
# 定义一个教学版 percentile 函数。
def simple_percentile(values, fraction):
    # 确保输入列表不为空。
    if not values:
        raise ValueError("values 不能为空")

    # 将样本从小到大排序，避免改变原始列表。
    ordered_values = sorted(values)

    # 根据 nearest-rank 思路计算数组位置。
    position = max(0, min(len(ordered_values) - 1, round(fraction * len(ordered_values)) - 1))

    # 返回对应位置的样本。
    return ordered_values[position]

# 计算稳定态平均值。
mean_ms = mean(stable_samples)

# 计算稳定态中位数，也就是 P50 的一种实现。
p50_ms = median(stable_samples)

# 计算稳定态 P95。
p95_ms = simple_percentile(stable_samples, 0.95)

# 打印三种读数。
print({"mean_ms": round(mean_ms, 4), "p50_ms": round(p50_ms, 4), "p95_ms": round(p95_ms, 4)})

# 三个延迟指标都必须为非负数。
assert mean_ms >= 0 and p50_ms >= 0 and p95_ms >= 0

{'mean_ms': 0.0312, 'p50_ms': 0.0293, 'p95_ms': 0.046}


### 如何解释三个数字

- Mean 适合看总体耗时，但可能受极端值影响。
- P50 表示一半请求不超过的耗时，接近典型请求。
- P95 表示 95% 请求不超过的耗时，能发现长尾。

不能把当前几毫秒的结果写成“系统生产性能”，因为数据规模、硬件、并发和模型路径都不同。

In [7]:
# 记录当前 Python 版本，方便复现实验环境。
python_version = sys.version.split()[0]

# 组合 benchmark 基线记录。
timing_record = {"query": "Chunk overlap", "top_k": 5, "warmup": 5, "iterations": 30, "python": python_version, "index_version": knowledge_base.index_version, "first_ms": first_ms, "mean_ms": mean_ms, "p50_ms": p50_ms, "p95_ms": p95_ms}

# 指定 Phase 3 基线文件路径。
timing_path = ROOT / "data" / "processed" / "phase3_timing_baseline.json"

# 保存计时条件和结果。
timing_path.write_text(json.dumps(timing_record, ensure_ascii=False, indent=2), encoding="utf-8")

# 打印交付路径。
print("已生成:", timing_path)

已生成: D:\code\codeByCursor\AI_EXAM\ai-search-rag-internship\data\processed\phase3_timing_baseline.json


## 本课验收

- [ ] 能解释为什么先 warmup 再测稳定态。
- [ ] 能说明 mean、P50、P95 的不同含义。
- [ ] 计时函数测的是实际搜索而不是空函数。
- [ ] 已保存条件、版本、迭代次数和结果。

## Boss Challenge：把 warmup 次数和测量次数各改一次，说明为什么冷启动和稳定延迟不能混报。

下面是**故意保持注释状态**的跟敲模板。请先自己写，再取消注释逐行运行；不要把它当成需要复制的答案。

In [8]:
# 第 1 行：先写出本挑战需要的新变量或新输入。
# challenge_input = ...

# 第 2 行：调用本课已经学会的函数或模块。
# challenge_result = ...

# 第 3 行：打印一个中间结果，先观察再下结论。
# print(challenge_result)

# 第 4 行：写一个断言，把你的理解变成机器可检查的条件。
# assert ...

## 作品检查站

作品不是‘我运行过代码’，而是别人可以在文件浏览器中找到、下一阶段可以读取、你能解释生成过程的证据。下面的检查只报告事实，不替你假装通关。

In [9]:
# 列出本关应该产生的作品路径。
quest_artifact_candidates = ['data/processed/phase3_timing_baseline.json']

# 把相对路径转换为项目根目录下的绝对路径。
quest_artifact_paths = [ROOT / path for path in quest_artifact_candidates]

# 只保留已经真正写入磁盘的作品。
quest_existing_artifacts = [str(path.relative_to(ROOT)) for path in quest_artifact_paths if path.is_file()]

# 保存一个不依赖外部服务的本关检查结果，方便复盘。
quest_checkpoint = {"stage": QUEST_STAGE, "existing_artifacts": quest_existing_artifacts}

# 打印检查结果，让学习者知道下一步是继续学习还是补交作品。
print("本关作品:", quest_existing_artifacts if quest_existing_artifacts else "还没有生成，请回到交付单元格")

本关作品: ['data\\processed\\phase3_timing_baseline.json']
